# 06 - Logistic Regression tuning, U.S. training only

This notebook tunes Logistic Regression on the official U.S. training rows with one-hot categorical preprocessing. All learned transformations are refit inside each CV training fold. The reserved U.S. IDs and labels are not loaded.


In [1]:
from pathlib import Path
import sys, warnings
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'preprocessing.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from utils.us_modeling import (CONSERVATIVE_POLICY, development_cv, load_us_training_data,
    run_randomized_search, scoring, search_result, conservative_pipeline)
SEED = 42
X_train, y_train, project_ids = load_us_training_data(ROOT)
assert len(project_ids) == 26945 and project_ids.is_unique
assert np.isfinite(y_train).all() and set(y_train.unique()) == {0, 1}
assert CONSERVATIVE_POLICY == dict(include_timing=False, include_flags=False, include_reward=False, exclude_renamed=True)
baseline = conservative_pipeline(LogisticRegression(max_iter=2000, solver='lbfgs'), 'onehot')
base_scores = cross_validate(baseline, X_train, y_train, cv=development_cv(), scoring=scoring(), n_jobs=1, return_train_score=True, error_score='raise')
baseline_row = {m: float(base_scores[f'test_{m}'].mean()) for m in scoring()}
baseline_row['roc_auc_sd'] = float(base_scores['test_roc_auc'].std())
print(f'official U.S. training rows: {len(X_train):,}; success rate: {y_train.mean():.4f}')
print('baseline:', {k: round(v, 4) for k, v in baseline_row.items()})


official U.S. training rows: 26,945; success rate: 0.5461
baseline: {'roc_auc': 0.7199, 'accuracy': 0.6684, 'f1': 0.7111, 'roc_auc_sd': 0.0055}


In [2]:
model = LogisticRegression(max_iter=2500, solver='lbfgs')
space = {
    'model__C': loguniform(1e-3, 1e2),
    'model__class_weight': [None, 'balanced'],
}
search, trials = run_randomized_search(X_train, y_train, model, 'onehot', space, n_iter=15)
result = search_result(search, trials, 'Logistic Regression')
result['baseline_roc_auc'] = baseline_row['roc_auc']
result_table = pd.DataFrame([result]).drop(columns=['best_params'])
display(result_table.round(4))
print('best parameters:', result['best_params'])
display(trials.head(10).drop(columns=['params']).round(4))


,model,cohort,feature_policy,cv,search_iterations,estimate,cv_roc_auc,cv_roc_auc_sd,train_roc_auc,overfit_gap,accuracy,f1,fit_seconds,elapsed_seconds,baseline_roc_auc
0,Logistic Regression,official U.S. training rows,"conservative: no timing, flags, rewards, or re...",5-fold StratifiedKFold shuffle=True random_sta...,15,ordinary tuning CV; optimistic selection estimate,0.7201,0.0054,0.7252,0.0051,0.6684,0.7114,0.2174,24.4363,0.7199


best parameters: {'C': np.float64(0.16949768237884735), 'class_weight': None}


,trial_rank,cv_roc_auc,cv_roc_auc_sd,train_roc_auc,cv_accuracy,cv_f1,mean_fit_seconds,overfit_gap,elapsed_seconds
0,1,0.7201,0.0054,0.7252,0.6684,0.7114,0.2174,0.0051,24.4363
1,2,0.7201,0.0054,0.7252,0.6685,0.7116,0.2196,0.0051,24.4363
2,3,0.7200,0.0053,0.7247,0.6678,0.7114,0.2139,0.0047,24.4363
3,4,0.7199,0.0054,0.7254,0.6686,0.7113,0.2535,0.0055,24.4363
4,5,0.7199,0.0055,0.7255,0.6686,0.7112,0.2225,0.0055,24.4363
5,6,0.7199,0.0055,0.7255,0.6686,0.7112,0.2413,0.0056,24.4363
6,7,0.7198,0.0055,0.7255,0.6630,0.6848,0.1965,0.0057,24.4363
7,8,0.7198,0.0056,0.7255,0.6631,0.6849,0.1883,0.0057,24.4363
8,9,0.7197,0.0052,0.7241,0.6624,0.6840,0.1649,0.0044,24.4363
9,10,0.7156,0.0043,0.7183,0.6592,0.6809,0.1770,0.0027,24.4363


In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(trials['mean_fit_seconds'], trials['cv_roc_auc'], c=trials['overfit_gap'], cmap='viridis')
ax.set_xscale('log'); ax.set_xlabel('Mean fit seconds'); ax.set_ylabel('Validation ROC-AUC'); ax.set_title('Logistic tuning cost and train-validation gap')
plt.tight_layout()
print(f"baseline AUC {baseline_row['roc_auc']:.4f}; tuned AUC {result['cv_roc_auc']:.4f}; difference {result['cv_roc_auc'] - baseline_row['roc_auc']:+.4f}")
print(f"best-trial train-validation gap: {result['overfit_gap']:.4f}")
print('The earlier 05a result used four extra candidates together. It is not an isolated feature-causality result and is not used here.')


baseline AUC 0.7199; tuned AUC 0.7201; difference +0.0002
best-trial train-validation gap: 0.0051
The earlier 05a result used four extra candidates together. It is not an isolated feature-causality result and is not used here.


The earlier 05a extension is kept as a separate exploratory comparison. This notebook ranks only conservative configurations. Search scores are tuning estimates, not independent generalization estimates.
